<a href="https://colab.research.google.com/github/chronoverse/Machine-Learning-for-Algorithmic-Trading-Second-Edition/blob/master/tablib_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

A notebook to demo all the indicators in ta-lib

1. READ EOD as a csv to df

In [9]:
# prompt: find version of python

!python --version


Python 3.10.12


In [10]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
import yaml
with open('/content/drive/My Drive/config.yaml', 'r') as fy:
  kv = yaml.safe_load(fy)
  API_TOKEN = kv['EODHDAPI']

In [13]:
url = 'https://anaconda.org/conda-forge/libta-lib/0.4.0/download/linux-64/libta-lib-0.4.0-h166bdaf_1.tar.bz2'
!curl -L $url | tar xj -C /usr/lib/x86_64-linux-gnu/ lib --strip-components=1
url = 'https://anaconda.org/conda-forge/ta-lib/0.4.19/download/linux-64/ta-lib-0.4.19-py310hde88566_4.tar.bz2'
!curl -L $url | tar xj -C /usr/local/lib/python3.10/dist-packages/ lib/python3.10/site-packages/talib --strip-components=3
import talib

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  4063    0  4063    0     0  20704      0 --:--:-- --:--:-- --:--:-- 20729
100  517k  100  517k    0     0  1309k      0 --:--:-- --:--:-- --:--:-- 1309k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  4071    0  4071    0     0  17289      0 --:--:-- --:--:-- --:--:-- 17323
100  392k  100  392k    0     0  1001k      0 --:--:-- --:--:-- --:--:-- 1001k


In [14]:
try:
    import talib
except ImportError:
    # https://stackoverflow.com/questions/49648391/how-to-install-ta-lib-in-google-colab
    url = 'https://anaconda.org/conda-forge/libta-lib/0.4.0/download/linux-64/libta-lib-0.4.0-h166bdaf_1.tar.bz2'
    !curl -L $url | tar xj -C /usr/lib/x86_64-linux-gnu/ lib --strip-components=1
    url = 'https://anaconda.org/conda-forge/ta-lib/0.4.19/download/linux-64/ta-lib-0.4.19-py310hde88566_4.tar.bz2'
    !curl -L $url | tar xj -C /usr/local/lib/python3.10/dist-packages/ lib/python3.10/site-packages/talib --strip-components=3
    import talib

In [ ]:
import pandas as pd
from datetime import datetime, date
import os
import time
import logging
logging.basicConfig(level=logging.DEBUG)

class TA():
  """Download EOD of SPY"""
  def __init__(self):
    self.__parq_df = None
    self.__base_url = "https://eodhd.com/api"
    self.__start_date = date(2007, 1, 1)
    self.__end_date: str = datetime.today()
    return

  def load_spy(self, parqfile: str = "GSPC.parquet"):
    """Load GSPC"""
    if os.path.exists(parqfile) and \
     ((time.time()-os.path.getmtime(parqfile))<3600):
      logging.debug(f"Loading from {parqfile}")
      self.__parq_df = pd.read_parquet(parqfile)

    else:
        df_dict = dict()  # an empty dict of dataframes
        rows2skip = 0
        for tick in ['GSPC.INDX']:
            logging.debug(f"Processing {tick}")
            # read the csv
            TO = self.__end_date.strftime("%Y-%m-%d")
            FROM = self.__start_date.strftime("%Y-%m-%d")
            url = (f"{self.__base_url}/eod/{tick}?"
                   f"to={TO}&from={FROM}&period=d"
                   f"fmt=csv&api_token={API_TOKEN}"
            )
            self.__parq_df = pd.read_csv(url, skiprows=rows2skip,
                                         parse_dates=['Date'])
            self.__parq_df.to_parquet(parqfile)
            logging.debug(f"Done processing {tick}")

            rows2skip = 1


    return self.__parq_df


In [ ]:
ta = TA()
gspc_df = ta.load_spy()

In [ ]:
gspc_df.info()

In [ ]:
!ls -l